# 01 — Homografia (Normalização de Perspectiva)

Transforma cada foto da mesa em uma **visão top-down padronizada** (800×400 px),
eliminando diferenças de ângulo e perspectiva entre as fotos do dataset.

**Como funciona:**
1. Você clica nas **4 quinas da mesa** em cada imagem (TL → TR → BR → BL)
2. O notebook calcula a matriz de homografia
3. Aplica `warpPerspective` e salva a imagem normalizada

**Entrada:** `data/converted/`  
**Saída:** `data/normalized/` (todas 800×400 px, mesma orientação)

---
**Ordem dos cliques:**
```
TL ─────────── TR
│               │
│               │
BL ─────────── BR
```

## Célula 1 — Monta o Drive e define caminhos

In [1]:
import os
import sys

NO_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ

if NO_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    CAMINHO_BASE = '/content/drive/MyDrive/Colab Notebooks/SinucaVision/data'
    !pip install opencv-python-headless matplotlib numpy -q
else:
    CAMINHO_BASE = os.path.join(os.path.dirname(os.getcwd()), 'data')

# Caminhos específicos da homografia
CAMINHO_CONVERTED  = os.path.join(CAMINHO_BASE, 'converted')
CAMINHO_NORMALIZED = os.path.join(CAMINHO_BASE, 'normalized')
CAMINHO_ANNOT      = os.path.join(CAMINHO_BASE, 'annotations')

os.makedirs(CAMINHO_CONVERTED,  exist_ok=True)
os.makedirs(CAMINHO_NORMALIZED, exist_ok=True)
os.makedirs(CAMINHO_ANNOT,      exist_ok=True)

# Dimensões fixas da saída top-down
LARGURA_MESA = 800
ALTURA_MESA  = 400

print(f'Ambiente:   {"Google Colab" if NO_COLAB else "Local"}')
print(f'Converted:  {CAMINHO_CONVERTED}')
print(f'Normalized: {CAMINHO_NORMALIZED}')
print(f'Annotations:{CAMINHO_ANNOT}')
print(f'Saída:      {LARGURA_MESA}×{ALTURA_MESA} px')

Ambiente:   Local
Converted:  /home/bridge/SinucaVision/data/converted
Normalized: /home/bridge/SinucaVision/data/normalized
Annotations:/home/bridge/SinucaVision/data/annotations
Saída:      800×400 px


## Célula 2 — Backend interativo

In [2]:
%matplotlib tk

ModuleNotFoundError: No module named 'tkinter'

## Célula 3 — Funções de homografia

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import json
import glob

def aplicar_homografia(img_bgr, pontos):
    pts_src = np.float32(pontos)
    pts_dst = np.float32([
        [0,            0           ],
        [LARGURA_MESA, 0           ],
        [LARGURA_MESA, ALTURA_MESA ],
        [0,            ALTURA_MESA ],
    ])
    M = cv2.getPerspectiveTransform(pts_src, pts_dst)
    return cv2.warpPerspective(img_bgr, M, (LARGURA_MESA, ALTURA_MESA))

def mostrar_com_pontos(img_rgb, pontos, titulo=''):
    labels = ['TL', 'TR', 'BR', 'BL']
    cores  = ['red', 'blue', 'green', 'orange']
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.imshow(img_rgb)
    ax.set_title(titulo, fontsize=12)
    ax.axis('off')
    for i, (x, y) in enumerate(pontos):
        ax.plot(x, y, 'o', color=cores[i], markersize=14)
        ax.annotate(labels[i], (x, y),
                    textcoords='offset points', xytext=(10, 10),
                    fontsize=13, color=cores[i], fontweight='bold')
    patches = [mpatches.Patch(color=c, label=l) for c, l in zip(cores, labels)]
    ax.legend(handles=patches, loc='upper right')
    plt.tight_layout()
    plt.show()

print('Funções carregadas.')

Funções carregadas.


## Célula 4 — Processamento interativo (uma imagem por vez)

Altere `NOME_IMAGEM` para a foto desejada e execute.  
Execute esta célula para cada imagem.
Clique nas **4 quinas da mesa** na ordem: **TL → TR → BR → BL**

> **Dica:** zoom na imagem antes de clicar para maior precisão.
> Use o botão de zoom do matplotlib (ícone de lupa) antes de coletar os pontos.

In [ ]:
NOME_IMAGEM = 'IMG_0640.JPG'

caminho_img   = os.path.join(CAMINHO_CONVERTED, NOME_IMAGEM)
caminho_annot = os.path.join(CAMINHO_ANNOT, NOME_IMAGEM.replace('.JPG', '.json'))
caminho_saida = os.path.join(CAMINHO_NORMALIZED, NOME_IMAGEM)

img_bgr = cv2.imread(caminho_img)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
h, w    = img_bgr.shape[:2]
print(f'Imagem: {NOME_IMAGEM} ({w}×{h} px)')

if os.path.exists(caminho_annot):
    with open(caminho_annot) as f:
        pontos = json.load(f)['quinas']
    print('Anotação já existente — pulando coleta de pontos.')
    mostrar_com_pontos(img_rgb, pontos, titulo=f'{NOME_IMAGEM} — pontos salvos')
else:
    print()
    print('Clique nas 4 quinas na ordem: TL → TR → BR → BL')
    print('Após o 4º clique, os pontos são salvos automaticamente.')

    fig, ax = plt.subplots(figsize=(14, 8))
    ax.imshow(img_rgb)
    ax.set_title(f'{NOME_IMAGEM} — clique: TL → TR → BR → BL', fontsize=12)
    ax.axis('off')
    plt.tight_layout()

    # Com %matplotlib widget o ginput funciona normalmente no JupyterLab
    pontos_raw = plt.ginput(4, timeout=0)  # timeout=0 espera indefinidamente
    plt.close()

    pontos = [[round(x, 1), round(y, 1)] for x, y in pontos_raw]
    print(f'Pontos coletados: TL={pontos[0]} TR={pontos[1]} BR={pontos[2]} BL={pontos[3]}')

    with open(caminho_annot, 'w') as f:
        json.dump({
            'imagem': NOME_IMAGEM,
            'resolucao_original': [w, h],
            'quinas': pontos,
            'ordem': ['TL', 'TR', 'BR', 'BL']
        }, f, indent=2)
    print(f'Anotação salva: {caminho_annot}')

Imagem: IMG_0640.JPG (4284×5712 px)
Anotação já existente — pulando coleta de pontos.


## Célula 5 — Processa todas as imagens.

Rode após ter anotado todas as imagens na Célula 4.  
Imagens com anotação já salva em `annotations/` são processadas sem abrir janela.  
Imagens já normalizadas em `normalized/` são puladas automaticamente.

In [ ]:
# Rode esta célula após a Célula 4 para ver e salvar a normalizada
with open(caminho_annot) as f:
    pontos = json.load(f)['quinas']

mostrar_com_pontos(img_rgb, pontos, titulo=f'{NOME_IMAGEM} — quinas selecionadas')

img_norm = aplicar_homografia(img_bgr, pontos)
cv2.imwrite(caminho_saida, img_norm)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
ax1.imshow(img_rgb)
ax1.set_title('Original', fontsize=11)
ax1.axis('off')
ax2.imshow(cv2.cvtColor(img_norm, cv2.COLOR_BGR2RGB))
ax2.set_title(f'Normalizada ({LARGURA_MESA}×{ALTURA_MESA} px)', fontsize=11)
ax2.axis('off')
plt.tight_layout()
plt.show()
print(f'Salva em: {caminho_saida}')

Salva em: /home/juliana/Projects/UFSC/visao/SinucaVision/data/normalized/IMG_0640.JPG


## Célula 6 — Visualiza todas as imagens normalizadas

In [ ]:
normalizadas = sorted(glob.glob(os.path.join(CAMINHO_NORMALIZED, '*.JPG')))

if not normalizadas:
    print('Nenhuma imagem normalizada encontrada. Execute as células anteriores.')
else:
    print(f'{len(normalizadas)} imagens normalizadas encontradas.')
    cols = 3
    rows = -(-len(normalizadas) // cols)

    fig, axes = plt.subplots(rows, cols, figsize=(15, 4 * rows))
    axes = axes.flatten()

    for ax, caminho in zip(axes, normalizadas):
        img = cv2.imread(caminho)
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(os.path.basename(caminho), fontsize=8)
        ax.axis('off')

    for ax in axes[len(normalizadas):]:
        ax.axis('off')

    plt.suptitle(
        f'Imagens Normalizadas — {LARGURA_MESA}×{ALTURA_MESA} px (top-down)',
        fontsize=13
    )
    plt.tight_layout()
    plt.show()

6 imagens normalizadas encontradas.


## Célula 7 — Batch: aplica homografia a todas as imagens anotadas

Lê todos os JSONs em `data/annotations/`, aplica a homografia e salva em `data/normalized/`.  
Imagens sem anotação são listadas ao final — use a Célula 4 para anotá-las.

In [ ]:
anotacoes = sorted(glob.glob(os.path.join(CAMINHO_ANNOT, '*.json')))
convertidas = sorted(
    glob.glob(os.path.join(CAMINHO_CONVERTED, '*.JPG')) +
    glob.glob(os.path.join(CAMINHO_CONVERTED, '*.jpg'))
)
nomes_convertidas = {os.path.splitext(os.path.basename(p))[0] for p in convertidas}

print(f'Anotações encontradas: {len(anotacoes)}')
print(f'Imagens em converted/: {len(convertidas)}')
print()

processadas, sem_anotacao = [], []

# Processa imagens com anotação
for caminho_json in anotacoes:
    nome_base = os.path.splitext(os.path.basename(caminho_json))[0]

    # Tenta .JPG e .jpg
    img_path = os.path.join(CAMINHO_CONVERTED, nome_base + '.JPG')
    if not os.path.exists(img_path):
        img_path = os.path.join(CAMINHO_CONVERTED, nome_base + '.jpg')
    if not os.path.exists(img_path):
        print(f'  [ERRO] Imagem não encontrada para {nome_base}')
        continue

    with open(caminho_json) as f:
        pontos = json.load(f)['quinas']

    img = cv2.imread(img_path)
    img_norm = aplicar_homografia(img, pontos)

    caminho_saida = os.path.join(CAMINHO_NORMALIZED, nome_base + '.JPG')
    cv2.imwrite(caminho_saida, img_norm)
    processadas.append(nome_base)
    print(f'  ✓  {nome_base}.JPG → normalized/')

# Lista imagens sem anotação
nomes_anotados = {os.path.splitext(os.path.basename(p))[0] for p in anotacoes}
sem_anotacao   = sorted(nomes_convertidas - nomes_anotados)

print(f'\nProcessadas: {len(processadas)}')
if sem_anotacao:
    print(f'Sem anotação ({len(sem_anotacao)}) — use a Célula 4:')
    for nome in sem_anotacao:
        print(f'  ⏳  {nome}')
else:
    print('Todas as imagens convertidas já têm anotação.')


Anotações encontradas: 6
Imagens em converted/: 13

  ✓  IMG_0640.JPG → normalized/
  ✓  IMG_0641.JPG → normalized/
  ✓  IMG_0642.JPG → normalized/
  ✓  IMG_0643.JPG → normalized/
  ✓  IMG_0644.JPG → normalized/
  ✓  IMG_0645.JPG → normalized/

Processadas: 6
Sem anotação (7) — use a Célula 4:
  ⏳  IMG_0646
  ⏳  IMG_0647
  ⏳  IMG_0648
  ⏳  IMG_0649
  ⏳  IMG_0650
  ⏳  IMG_0651
  ⏳  IMG_0652
